In [16]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/processed/dashboard_data.csv")
conn = sqlite3.connect("../data/processed/churn.db")

df.to_sql("customers", conn, if_exists="replace", index=False)

print("Table created successfully")

Table created successfully


In [17]:
query = """
SELECT 
    Contract,
    ROUND(AVG(Churn), 3) AS churn_rate,
    COUNT(*) AS customers
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

pd.read_sql(query, conn)

,Contract,churn_rate,customers
0,Month-to-month,0.426,3853
1,One year,0.113,1473
2,Two year,0.028,1695


In [18]:
query = """
SELECT 
    SUM(customer_value * churn_probability) AS revenue_at_risk
FROM customers;
"""

pd.read_sql(query, conn)

,revenue_at_risk
0,6.861850e+06


In [19]:
query = """
SELECT *
FROM customers
WHERE churn_probability > 0.6
AND customer_value > (
    SELECT AVG(customer_value) FROM customers
)
ORDER BY churn_probability DESC;
"""

pd.read_sql(query, conn).head()

,customerID,churn_probability,risk_segment,value_segment,customer_value,expected_saved,cost,profit,roi_per_customer,Churn,Contract,InternetService,MonthlyCharges
0,4815-YOSUK,0.979483,High Risk,High,3272.920439,1602.885556,400,1202.885556,3.007214,1,Month-to-month,Fiber optic,100.80
1,4425-OWHWB,0.978983,High Risk,High,3037.516935,1486.839345,400,1086.839345,2.717098,1,Month-to-month,Fiber optic,93.55
2,0511-JTEOY,0.978793,High Risk,High,3052.128187,1493.701608,400,1093.701608,2.734254,1,Month-to-month,Fiber optic,94.00
3,2883-ILGWO,0.978619,High Risk,High,3087.844581,1510.911270,400,1110.911270,2.777278,1,Month-to-month,Fiber optic,95.10
4,5501-TVMGM,0.974072,High Risk,High,3047.257770,1484.123939,400,1084.123939,2.710310,1,Month-to-month,Fiber optic,93.85


In [20]:
query = """
SELECT 
    risk_segment,
    ROUND(AVG(roi_per_customer), 2) AS avg_roi,
    COUNT(*) AS customers
FROM customers
GROUP BY risk_segment
ORDER BY avg_roi DESC;
"""

pd.read_sql(query, conn)

,risk_segment,avg_roi,customers
0,High Risk,3.60,1690
1,Medium Risk,2.13,1781
2,Low Risk,-0.18,3550


In [21]:
query = """
SELECT 
    value_segment,
    SUM(cost) AS total_cost,
    SUM(expected_saved) AS total_revenue,
    ROUND((SUM(expected_saved) - SUM(cost)) / SUM(cost), 2) AS roi
FROM customers
GROUP BY value_segment;
"""

pd.read_sql(query, conn)

,value_segment,total_cost,total_revenue,roi
0,High,845750,1.866192e+06,1.21
1,Low,234200,3.455351e+05,0.48
2,Medium,410550,1.219198e+06,1.97


In [23]:
query = """
SELECT 
    customerID,
    churn_probability,
    customer_value,
    roi_per_customer
FROM customers
ORDER BY roi_per_customer DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customerID,churn_probability,customer_value,roi_per_customer
0,6754-LZUKA,0.936588,2282.602250,9.689286
1,4523-WXCEF,0.936911,2279.355306,9.677770
2,5103-MHMHY,0.936972,2277.731833,9.670852
3,4307-KWMXE,0.936053,2274.484888,9.645189
4,4514-GFCFI,0.937213,2271.237943,9.643164
5,5088-QZLRL,0.937333,2267.990998,9.629310
6,1135-HIORI,0.936418,2264.744054,9.603741
7,8679-LZBMD,0.937572,2261.497109,9.601583
8,7109-MFBYV,0.938509,2258.250164,9.596941
9,8345-MVDYC,0.937632,2259.873636,9.594648
